# Gaussian scatterer superposition test

Builds two different Gaussian scattering foils, runs a 1&nbsp;mm-sigma pencil
beam through each separately, then builds a third scatterer whose thickness
profile is the pointwise **sum** of the first two, and runs the same beam
through that. Compares the combined scatterer's dose map against the
elementwise sum of the two individual dose maps.

Physically these are *not* expected to match exactly: multiple Coulomb
scattering angle (Highland formula) scales roughly with `sqrt(thickness)`,
not thickness itself, so scattering through a foil of combined thickness
`t1(r) + t2(r)` is not the same as adding the scattering distributions from
two foils of thickness `t1(r)` and `t2(r)` individually. This notebook
quantifies how far off that linear approximation is for this shape/material/
energy combination.

Setup mirrors `simulate_PDD_CLEAR.ipynb` / `CLARA_design.ipynb`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from topasToDose import getDosemap
from partrec_gaussian_optimiser_utils import partrec_gaussian_optimiser_utils

## Config

In [ ]:
# ---- beam ----
E = 200               # MeV
N_particles = int(2e5)
beam_sigma = 1         # mm, "1mm radius" Gaussian beam (position spread)
beam_sigma_p = 0.01    # mrad, ~parallel pencil beam

# ---- scatterers (foil material + geometry) ----
material = "PEEK"
scatterer_position = 100   # mm, z of the scatterer's downstream/base face
radius = 5                 # mm, SAME for both scatterers so their radial grids line up for summing
N_slices = 60               # finer than before to resolve the narrow spike below

# broad, low pedestal: large effective sigma (2.5*1.8=4.5mm) relative to radius, low peak
scatterer1 = dict(max_thickness=0.3, convolution_factor=1.8)
# narrow, tall spike: small effective sigma (2.5*0.15=0.375mm), high peak
scatterer2 = dict(max_thickness=1.6, convolution_factor=0.15)
# summed shape is a sharp spike sitting on a broad shallow dome -- deliberately
# not well-approximated by a single Gaussian, unlike the first version of this test

# ---- dose scorer ----
tank_position = 600   # mm, upstream face of the (thin) scoring tank
tank_depth = 1          # mm, single Z bin
tank_width = 60         # mm, scorer full width in x and y (wider: thicker/sharper combined foil scatters more)
tank_bins = 150         # keeps the same 0.4mm/bin resolution as before

acChargenC = 1  # nC -- arbitrary common reference charge, same for all three runs so dose maps are directly comparable

## Helper: arbitrary-profile scatterer

`add_gaussian_scatterer` on `partrec_gaussian_optimiser_utils` hardcodes its slice
names (`slice1`, `slice2`, ...), so two calls in the same script would collide.
`gaussian_height_profile` reproduces its radial thickness profile as plain arrays,
and `add_stacked_scatterer` builds a scatterer from *any* profile (under its own
name prefix) using the same stack-of-concentric-cylinders technique -- so the same
helper builds all three scatterers below, including the summed one.

In [ ]:
def gaussian_height_profile(radius, max_thickness, convolution_factor, N_slices):
    """Radial thickness profile of a Gaussian scatterer -- same x/y construction as
    partrec_gaussian_optimiser_utils.add_gaussian_scatterer, returned as arrays.
    x[i] is a radius (mm); y[i] is the cumulative height (mm) built up out to that
    radius, increasing from ~0 at the outer edge to max_thickness at the centre."""
    sigma = radius / 2
    step = radius / N_slices
    x = np.arange(-(radius + step), 0, step=step)
    y = norm.pdf(x, 0, sigma * convolution_factor)
    y = y - y.min()
    y = y * (max_thickness / y.max())
    return x, y


def add_stacked_scatterer(setup, x, y, material, position, name_prefix):
    """Build a solid of revolution from a radial height profile (x, y) using the same
    stack-of-concentric-cylinders construction as add_gaussian_scatterer, but under a
    caller-chosen name_prefix so several scatterers can coexist in one TOPAS script."""
    for i in range(1, len(y)):
        L = y[i] - y[i - 1]
        if L <= 0:
            continue
        sname = f"{name_prefix}{i}"
        # matches add_gaussian_scatterer's TransZ = -(position - y[i-1] - L/2);
        # add_cylinder itself adds +thickness/2, so pass position - y[i-1] - L
        setup.add_cylinder(sname, L, 0, abs(x[i]), material, position - y[i - 1] - L)

## Scatterer 1

In [ ]:
output_filename1 = "gauss_test_scatterer1"

setup1 = partrec_gaussian_optimiser_utils()
setup1.generate_phsp_beam(beam_sigma, beam_sigma, beam_sigma_p, beam_sigma_p, E, 0, N_particles)

x1, y1 = gaussian_height_profile(radius, scatterer1["max_thickness"], scatterer1["convolution_factor"], N_slices)
add_stacked_scatterer(setup1, x1, y1, material, scatterer_position, name_prefix="s1_slice")

setup1.add_tank_bins(tank_position, tank_depth, tank_bins, tank_bins, 1, output_filename1, width=tank_width)
setup1.run_topas(view_setup=False)

## Scatterer 2 (different shape)

In [ ]:
output_filename2 = "gauss_test_scatterer2"

setup2 = partrec_gaussian_optimiser_utils()
setup2.generate_phsp_beam(beam_sigma, beam_sigma, beam_sigma_p, beam_sigma_p, E, 0, N_particles)

x2, y2 = gaussian_height_profile(radius, scatterer2["max_thickness"], scatterer2["convolution_factor"], N_slices)
add_stacked_scatterer(setup2, x2, y2, material, scatterer_position, name_prefix="s2_slice")

setup2.add_tank_bins(tank_position, tank_depth, tank_bins, tank_bins, 1, output_filename2, width=tank_width)
setup2.run_topas(view_setup=False)

## Combined scatterer (sum of the two shapes)

In [ ]:
output_filename3 = "gauss_test_combined"

x_combined, y1_check = gaussian_height_profile(radius, scatterer1["max_thickness"], scatterer1["convolution_factor"], N_slices)
_, y2_check = gaussian_height_profile(radius, scatterer2["max_thickness"], scatterer2["convolution_factor"], N_slices)
y_combined = y1_check + y2_check  # pointwise sum of the two thickness profiles

plt.plot(np.abs(x_combined), y1_check, label="Scatterer 1")
plt.plot(np.abs(x_combined), y2_check, label="Scatterer 2")
plt.plot(np.abs(x_combined), y_combined, label="Combined (sum)")
plt.xlabel("radius (mm)")
plt.ylabel("thickness (mm)")
plt.legend()
plt.grid(alpha=0.3)
plt.title("Radial thickness profiles")
plt.show()

setup3 = partrec_gaussian_optimiser_utils()
setup3.generate_phsp_beam(beam_sigma, beam_sigma, beam_sigma_p, beam_sigma_p, E, 0, N_particles)

add_stacked_scatterer(setup3, x_combined, y_combined, material, scatterer_position, name_prefix="sum_slice")

setup3.add_tank_bins(tank_position, tank_depth, tank_bins, tank_bins, 1, output_filename3, width=tank_width)
setup3.run_topas(view_setup=False)

## Load the three dose maps

In [ ]:
dose_file1 = f"DoseAtTank{tank_depth}_{output_filename1}.csv"
dose_file2 = f"DoseAtTank{tank_depth}_{output_filename2}.csv"
dose_file3 = f"DoseAtTank{tank_depth}_{output_filename3}.csv"

x, y, doseMap1 = getDosemap(dose_file1, N_particles, tank_depth, output_filename1, acChargenC=acChargenC, plot=False)
_, _, doseMap2 = getDosemap(dose_file2, N_particles, tank_depth, output_filename2, acChargenC=acChargenC, plot=False)
_, _, doseMap_combined = getDosemap(dose_file3, N_particles, tank_depth, output_filename3, acChargenC=acChargenC, plot=False)

doseMap_superposed = doseMap1 + doseMap2

## Compare: combined-shape scatterer vs. summed individual dose maps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, data, title in zip(
    axes,
    [doseMap_superposed, doseMap_combined, doseMap_combined - doseMap_superposed],
    ["Scatterer1 + Scatterer2\n(summed dose maps)", "Combined-shape scatterer", "Difference\n(combined - summed)"],
):
    im = ax.imshow(data, extent=[x.min(), x.max(), y.min(), y.max()], origin="lower", cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("X (mm)")
    ax.set_ylabel("Y (mm)")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

In [ ]:
cx_idx, cy_idx = doseMap_combined.shape[1] // 2, doseMap_combined.shape[0] // 2

fig, (ax_x, ax_y) = plt.subplots(1, 2, figsize=(12, 5))
ax_x.plot(x, doseMap_superposed[cy_idx, :], "o", markersize=3, label="Scatterer1 + Scatterer2 (summed)")
ax_x.plot(x, doseMap_combined[cy_idx, :], "-", label="Combined-shape scatterer")
ax_x.set_xlabel("X (mm)")
ax_x.set_ylabel("Dose (Gy)")
ax_x.legend()
ax_x.grid(alpha=0.3)

ax_y.plot(y, doseMap_superposed[:, cx_idx], "o", markersize=3, label="Scatterer1 + Scatterer2 (summed)")
ax_y.plot(y, doseMap_combined[:, cx_idx], "-", label="Combined-shape scatterer")
ax_y.set_xlabel("Y (mm)")
ax_y.set_ylabel("Dose (Gy)")
ax_y.legend()
ax_y.grid(alpha=0.3)
plt.tight_layout()
plt.show()

peak_ref = doseMap_superposed.max()
mask = doseMap_superposed > 0.01 * peak_ref  # ignore near-zero tails where relative error is meaningless
pct_diff = np.full_like(doseMap_combined, np.nan)
pct_diff[mask] = (doseMap_combined[mask] - doseMap_superposed[mask]) / doseMap_superposed[mask] * 100

print(f"Max dose:  superposed={doseMap_superposed.max():.4g} Gy   combined={doseMap_combined.max():.4g} Gy")
print(f"Integrated dose ratio (combined / superposed): {doseMap_combined.sum() / doseMap_superposed.sum():.4f}")
print(f"Max |% difference| (combined vs. superposed):  {np.nanmax(np.abs(pct_diff)):.2f}%")

## Why the naive sum comparison is confounded

From the run above: peak dose summed = 22.64 Gy vs. combined = 7.56 Gy
(~3x lower), and the integrated dose ratio (combined / summed) is ~0.50.

That factor of ~2 in integrated dose is mostly a bookkeeping artefact, not
new scattering physics: `doseMap_superposed = doseMap1 + doseMap2` adds two
*independent* 1 nC beam deliveries (2 nC total), while `doseMap_combined` is
a single 1 nC delivery through one scatterer. The two maps were never on the
same total-charge footing, so a ~0.5 integrated ratio is close to what you'd
expect from that bookkeeping alone, before any scattering physics enters.

The physically meaningful test is whether the *shape* of `doseMap_combined`
matches what you'd get from combining the two scatterers' angular kernels in
quadrature (Highland: scattering-angle variance adds with thickness) -- i.e.
whether it looks like the **convolution** of the two individual
(charge-normalized) dose maps, rather than their sum.

In [ ]:
from scipy.signal import fftconvolve

# unit-integral "kernels" -- the spatial spread each scatterer alone produces,
# with the total-charge/amplitude scale divided out
kernel1 = doseMap1 / doseMap1.sum()
kernel2 = doseMap2 / doseMap2.sum()

# combining two independent scattering kernels in sequence is a convolution
# (Highland: variance adds with thickness -- a random-walk/diffusion picture,
# same as passing through one slab of thickness t1+t2)
kernel_conv = fftconvolve(kernel1, kernel2, mode="same")
kernel_conv = np.clip(kernel_conv, 0, None)
kernel_conv /= kernel_conv.sum()

# scale the predicted shape to the combined-scatterer run's own total dose,
# so we're comparing shape, not total delivered charge
doseMap_predicted = kernel_conv * doseMap_combined.sum()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, data, title in zip(
    axes,
    [doseMap_predicted, doseMap_combined, doseMap_combined - doseMap_predicted],
    ["Convolution of scatterer1 & 2\n(shape only, scaled to combined's total dose)",
     "Combined-shape scatterer (actual)",
     "Difference\n(actual - convolution prediction)"],
):
    im = ax.imshow(data, extent=[x.min(), x.max(), y.min(), y.max()], origin="lower", cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("X (mm)")
    ax.set_ylabel("Y (mm)")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

fig, (ax_x, ax_y) = plt.subplots(1, 2, figsize=(12, 5))
ax_x.plot(x, doseMap_predicted[cy_idx, :], "o", markersize=3, label="Convolution prediction")
ax_x.plot(x, doseMap_combined[cy_idx, :], "-", label="Combined-shape scatterer (actual)")
ax_x.set_xlabel("X (mm)")
ax_x.set_ylabel("Dose (Gy)")
ax_x.legend()
ax_x.grid(alpha=0.3)

ax_y.plot(y, doseMap_predicted[:, cx_idx], "o", markersize=3, label="Convolution prediction")
ax_y.plot(y, doseMap_combined[:, cx_idx], "-", label="Combined-shape scatterer (actual)")
ax_y.set_xlabel("Y (mm)")
ax_y.set_ylabel("Dose (Gy)")
ax_y.legend()
ax_y.grid(alpha=0.3)
plt.tight_layout()
plt.show()

peak_ref = doseMap_predicted.max()
mask = doseMap_predicted > 0.01 * peak_ref
pct_diff_conv = np.full_like(doseMap_combined, np.nan)
pct_diff_conv[mask] = (doseMap_combined[mask] - doseMap_predicted[mask]) / doseMap_predicted[mask] * 100

print(f"Peak dose:  convolution prediction={doseMap_predicted.max():.4g} Gy   actual combined={doseMap_combined.max():.4g} Gy")
print(f"Max |% difference| (actual vs. convolution prediction): {np.nanmax(np.abs(pct_diff_conv)):.2f}%")